# Retail Sales SQL Analysis

## Project Overview

This project analyzes retail sales data using SQL and SQLite.

## Objectives
- Analyze sales and profit performance.
- Identify profitable and loss-making categories.
- Evaluate discount impact on profitability.
- Compare regional performance.
- Analyze shipping methods.
- Generate business insights.

## Technologies
- Python
- Pandas
- SQLite
- SQL
- Jupyter Notebook

In [1]:
import pandas as pd
import sqlite3

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df_clean = pd.read_csv("retail_cleaned.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df_clean.shape)

df_clean.head()

Dataset loaded successfully!
Dataset Shape: (9977, 14)


,Ship_Mode,Segment,Country,City,State,Postal_Code,Region,Category,Sub_Category,Sales,Quantity,Discount,Profit,Discount_Group
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.96,2.0,0.00,41.91,0-10%
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3.0,0.00,219.58,0-10%
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.62,2.0,0.00,6.87,0-10%
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.58,5.0,0.45,-383.03,40-50%
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.37,2.0,0.20,2.52,10-20%


In [3]:
print("Dataset Shape:", df_clean.shape)

print("\nColumn Names:")
print(df_clean.columns.tolist())

print("\nMissing Values:")
print(df_clean.isnull().sum().sum())

print("\nDuplicate Records:")
print(df_clean.duplicated().sum())

Dataset Shape: (9977, 14)

Column Names:
['Ship_Mode', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Category', 'Sub_Category', 'Sales', 'Quantity', 'Discount', 'Profit', 'Discount_Group']

Missing Values:
0

Duplicate Records:
0


In [4]:
import sqlite3

conn = sqlite3.connect("retail_sales.db")

df_clean.to_sql(
    "retail_sales",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully!")
print("Table name: retail_sales")

SQLite database created successfully!
Table name: retail_sales


In [5]:
query = """
SELECT *
FROM retail_sales
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,Ship_Mode,Segment,Country,City,State,Postal_Code,Region,Category,Sub_Category,Sales,Quantity,Discount,Profit,Discount_Group
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.96,2.0,0.00,41.91,0-10%
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3.0,0.00,219.58,0-10%
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.62,2.0,0.00,6.87,0-10%
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.58,5.0,0.45,-383.03,40-50%
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.37,2.0,0.20,2.52,10-20%


In [6]:
query = """
SELECT COUNT(*) AS Total_Records
FROM retail_sales;
"""

pd.read_sql_query(query, conn)

,Total_Records
0,9977


In [7]:
query = """
SELECT
    Category,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY Category
ORDER BY Total_Profit DESC;
"""

category_analysis = pd.read_sql_query(query, conn)

category_analysis

,Category,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,Technology,1847,836154.10,145455.66,78.75
1,Office Supplies,6012,718735.21,122364.75,20.35
2,Furniture,2118,741306.50,18421.79,8.70


In [8]:
query = """
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(
        (SUM(Profit) * 100.0 / SUM(Sales)), 2
    ) AS Profit_Margin_Percentage
FROM retail_sales
GROUP BY Category
ORDER BY Profit_Margin_Percentage DESC;
"""

profit_margin = pd.read_sql_query(query, conn)

profit_margin

,Category,Total_Sales,Total_Profit,Profit_Margin_Percentage
0,Technology,836154.10,145455.66,17.40
1,Office Supplies,718735.21,122364.75,17.03
2,Furniture,741306.50,18421.79,2.49


In [9]:
query = """
SELECT
    Region,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY Region
ORDER BY Total_Profit DESC;
"""

region_analysis = pd.read_sql_query(query, conn)

region_analysis

,Region,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,West,3193,725255.74,108330.15,33.93
1,East,2845,678435.32,91506.37,32.16
2,South,1620,391721.90,46749.71,28.86
3,Central,2319,500782.85,39655.97,17.10


In [10]:
query = """
SELECT
    COUNT(*) AS Total_Loss_Records,
    ROUND(SUM(Sales), 2) AS Sales_From_Loss_Records,
    ROUND(SUM(Profit), 2) AS Total_Loss
FROM retail_sales
WHERE Profit < 0;
"""

loss_analysis = pd.read_sql_query(query, conn)

loss_analysis

,Total_Loss_Records,Sales_From_Loss_Records,Total_Loss
0,1869,468422.45,-156113.56


In [11]:
query = """
SELECT
    Category,
    COUNT(*) AS Loss_Records,
    ROUND(SUM(Sales), 2) AS Sales_From_Loss_Records,
    ROUND(SUM(Profit), 2) AS Total_Loss
FROM retail_sales
WHERE Profit < 0
GROUP BY Category
ORDER BY Total_Loss ASC;
"""

category_loss = pd.read_sql_query(query, conn)

category_loss

,Category,Loss_Records,Sales_From_Loss_Records,Total_Loss
0,Furniture,713,257604.36,-60924.33
1,Office Supplies,885,91605.11,-56609.19
2,Technology,271,119212.98,-38580.04


In [12]:
query = """
SELECT
    Discount,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY Discount
ORDER BY Discount;
"""

discount_analysis = pd.read_sql_query(query, conn)

discount_analysis

,Discount,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,0.00,4787,1087277.56,320844.68,67.02
1,0.10,94,54369.30,9029.21,96.06
2,0.15,52,27558.59,1418.98,27.29
3,0.20,3653,764504.86,90307.47,24.72
4,0.30,226,102945.39,-10357.28,-45.83
5,0.32,27,14493.45,-2391.16,-88.56
6,0.40,206,116417.83,-23057.08,-111.93
7,0.45,11,5484.98,-2493.12,-226.65
8,0.50,66,58918.65,-20506.51,-310.70
9,0.60,138,6644.68,-5944.64,-43.08


In [13]:
query = """
SELECT
    Discount_Group,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY Discount_Group
ORDER BY Total_Profit DESC;
"""

discount_group_analysis = pd.read_sql_query(query, conn)

discount_group_analysis

,Discount_Group,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,0-10%,4881,1141646.86,329873.89,67.58
1,10-20%,3705,792063.45,91726.45,24.76
2,20-30%,226,102945.39,-10357.28,-45.83
3,40-50%,77,64403.63,-22999.63,-298.70
4,30-40%,233,130911.28,-25448.24,-109.22
5,50%+,855,64225.20,-76552.99,-89.54


In [14]:
query = """
SELECT
    Sub_Category,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM retail_sales
GROUP BY Sub_Category
ORDER BY Total_Profit DESC;
"""

subcategory_analysis = pd.read_sql_query(query, conn)

subcategory_analysis

,Sub_Category,Total_Records,Total_Sales,Total_Profit
0,Copiers,68,149528.01,55617.90
1,Phones,889,330007.10,44516.25
2,Accessories,775,167380.31,41936.78
3,Paper,1359,78224.18,33944.02
4,Binders,1522,203409.21,30227.88
5,Chairs,615,327777.79,26567.11
6,Storage,846,223843.59,21279.05
7,Appliances,466,107532.14,18138.07
8,Furnishings,956,91682.98,13052.83
9,Envelopes,254,16476.38,6964.10


In [15]:
query = """
SELECT
    Sub_Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM retail_sales
GROUP BY Sub_Category
ORDER BY Total_Profit DESC
LIMIT 10;
"""

top_profitable = pd.read_sql_query(query, conn)

top_profitable

,Sub_Category,Total_Sales,Total_Profit
0,Copiers,149528.01,55617.90
1,Phones,330007.10,44516.25
2,Accessories,167380.31,41936.78
3,Paper,78224.18,33944.02
4,Binders,203409.21,30227.88
5,Chairs,327777.79,26567.11
6,Storage,223843.59,21279.05
7,Appliances,107532.14,18138.07
8,Furnishings,91682.98,13052.83
9,Envelopes,16476.38,6964.10


In [16]:
query = """
SELECT
    Sub_Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Loss
FROM retail_sales
GROUP BY Sub_Category
HAVING SUM(Profit) < 0
ORDER BY Total_Loss ASC;
"""

loss_subcategories = pd.read_sql_query(query, conn)

loss_subcategories

,Sub_Category,Total_Sales,Total_Loss
0,Tables,206965.68,-17725.59
1,Bookcases,114880.05,-3472.56
2,Supplies,46673.52,-1188.99


In [17]:
query = """
SELECT
    Ship_Mode,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY Ship_Mode
ORDER BY Total_Profit DESC;
"""

shipping_analysis = pd.read_sql_query(query, conn)

shipping_analysis

,Ship_Mode,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,Standard Class,5955,1357316.69,163969.89,27.53
1,Second Class,1943,459176.92,57446.51,29.57
2,First Class,1537,351380.48,48953.77,31.85
3,Same Day,542,128321.72,15872.03,29.28


In [18]:
query = """
SELECT
    Region,
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    RANK() OVER (
        PARTITION BY Region
        ORDER BY SUM(Sales) DESC
    ) AS Sales_Rank
FROM retail_sales
GROUP BY Region, Category
ORDER BY Region, Sales_Rank;
"""

ranking_analysis = pd.read_sql_query(query, conn)

ranking_analysis

,Region,Category,Total_Sales,Sales_Rank
0,Central,Technology,170416.29,1
1,Central,Office Supplies,166959.27,2
2,Central,Furniture,163407.29,3
3,East,Technology,264974.04,1
4,East,Furniture,208009.80,2
5,East,Office Supplies,205451.48,3
6,South,Technology,148771.91,1
7,South,Office Supplies,125651.31,2
8,South,Furniture,117298.68,3
9,West,Furniture,252590.73,1


In [19]:
query = """
SELECT
    Category,
    Sub_Category,
    ROUND(Sales, 2) AS Sales,
    ROUND(Profit, 2) AS Profit
FROM retail_sales
WHERE Sales > (
    SELECT AVG(Sales)
    FROM retail_sales
)
ORDER BY Sales DESC
LIMIT 10;
"""

above_average_sales = pd.read_sql_query(query, conn)

above_average_sales

,Category,Sub_Category,Sales,Profit
0,Technology,Machines,22638.48,-1811.08
1,Technology,Copiers,17499.95,8399.98
2,Technology,Copiers,13999.96,6719.98
3,Technology,Copiers,11199.97,3919.99
4,Technology,Copiers,10499.97,5039.99
5,Office Supplies,Binders,9892.74,4946.37
6,Office Supplies,Binders,9449.95,4630.48
7,Technology,Machines,9099.93,2365.98
8,Technology,Machines,8749.95,2799.98
9,Technology,Copiers,8399.98,1120.00


In [20]:
query = """
SELECT
    r.Region,
    r.Category,
    ROUND(SUM(r.Sales), 2) AS Category_Sales,
    ROUND(s.Total_Sales, 2) AS Region_Total_Sales
FROM retail_sales r
INNER JOIN region_summary s
    ON r.Region = s.Region
GROUP BY
    r.Region,
    r.Category,
    s.Total_Sales
ORDER BY
    r.Region,
    Category_Sales DESC;
"""

join_analysis = pd.read_sql_query(query, conn)

join_analysis

,Region,Category,Category_Sales,Region_Total_Sales
0,Central,Technology,170416.29,500782.85
1,Central,Office Supplies,166959.27,500782.85
2,Central,Furniture,163407.29,500782.85
3,East,Technology,264974.04,678435.32
4,East,Furniture,208009.80,678435.32
5,East,Office Supplies,205451.48,678435.32
6,South,Technology,148771.91,391721.90
7,South,Office Supplies,125651.31,391721.90
8,South,Furniture,117298.68,391721.90
9,West,Furniture,252590.73,725255.74


In [21]:
query = """
SELECT
    Segment,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY Segment
ORDER BY Total_Profit DESC;
"""

segment_analysis = pd.read_sql_query(query, conn)

segment_analysis

,Segment,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,Consumer,5183,1160832.77,134007.56,25.86
1,Corporate,3015,706070.21,91955.30,30.50
2,Home Office,1779,429292.83,60279.34,33.88


In [22]:
query = """
SELECT
    State,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY State
ORDER BY Total_Sales DESC
LIMIT 10;
"""

state_analysis = pd.read_sql_query(query, conn)

state_analysis

,State,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,California,1996,457576.32,76331.00,38.24
1,New York,1127,310827.08,74015.55,65.67
2,Texas,983,170124.48,-25750.91,-26.20
3,Washington,502,138560.83,33368.29,66.47
4,Pennsylvania,586,116496.47,-15565.48,-26.56
5,Florida,383,89473.73,-3399.25,-8.88
6,Illinois,491,80162.60,-12601.65,-25.67
7,Ohio,468,77976.84,-16959.31,-36.24
8,Michigan,254,75879.64,24428.05,96.17
9,Virginia,224,70636.72,18598.00,83.03


In [23]:
query = """
SELECT
    State,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(AVG(Profit), 2) AS Average_Profit
FROM retail_sales
GROUP BY State
ORDER BY Total_Profit DESC
LIMIT 10;
"""

top_profitable_states = pd.read_sql_query(query, conn)

top_profitable_states

,State,Total_Records,Total_Sales,Total_Profit,Average_Profit
0,California,1996,457576.32,76331.00,38.24
1,New York,1127,310827.08,74015.55,65.67
2,Washington,502,138560.83,33368.29,66.47
3,Michigan,254,75879.64,24428.05,96.17
4,Virginia,224,70636.72,18598.00,83.03
5,Indiana,149,53555.36,18382.97,123.38
6,Georgia,184,49095.84,16250.08,88.32
7,Kentucky,139,36591.75,11199.70,80.57
8,Minnesota,89,29863.15,10823.22,121.61
9,Delaware,96,27451.07,9977.37,103.93


In [24]:
query = """
SELECT
    Category,
    COUNT(*) AS Total_Records,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS Loss_Records,
    ROUND(
        SUM(Profit) * 100.0 / SUM(Sales), 2
    ) AS Profit_Margin_Percentage
FROM retail_sales
GROUP BY Category
ORDER BY Total_Profit DESC;
"""

final_summary = pd.read_sql_query(query, conn)

final_summary

,Category,Total_Records,Total_Sales,Total_Profit,Loss_Records,Profit_Margin_Percentage
0,Technology,1847,836154.10,145455.66,271,17.40
1,Office Supplies,6012,718735.21,122364.75,885,17.03
2,Furniture,2118,741306.50,18421.79,713,2.49


## Project Conclusion

This project analyzed 9,977 retail sales records using Python, Pandas, SQLite, and SQL.

### Key Findings
- Technology generated the highest total profit of $145,455.66.
- Furniture recorded a low profit margin of 2.49%, despite generating $741,306.50 in sales.
- The West region generated the highest regional sales and profit.
- Higher discount levels were associated with lower profitability in this dataset.
- California generated the highest total sales and total profit among the states.
- Home Office had the highest average profit per record among customer segments.

### SQL Concepts Practiced
- SELECT, WHERE, GROUP BY, ORDER BY
- Aggregate functions: SUM(), COUNT(), AVG()
- HAVING and subqueries
- INNER JOIN
- CASE WHEN
- Window functions: RANK()

### Business Recommendations
- Review Furniture pricing and discount strategies.
- Investigate loss-making transactions.
- Focus on profitable products and regions.
- Monitor discount levels to protect profit margins.

### Conclusion
This project demonstrates the use of SQL for data analysis, business performance evaluation, and extracting actionable insights from retail sales data.